# K-Nearest Neighbors (K-NN) em datasets com class imbalance

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from math import sqrt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report,
)

## Implementação KNN from scratch

In [ ]:
class KNN:
    def __init__(self, k=5):
        self.k = k

    def fit(self, X_train, y_train):
        self.x_train = X_train
        self.y_train = y_train

    def calculate_euclidean(self, sample1, sample2):
        distance = 0.0
        for i in range(len(sample1)):
            distance += (sample1[i] - sample2[i]) ** 2
        return sqrt(distance)

    def nearest_neighbors(self, test_sample):
        distances = []
        for i in range(len(self.x_train)):
            distances.append((self.y_train[i], self.calculate_euclidean(self.x_train[i], test_sample)))
        distances.sort(key=lambda x: x[1])
        neighbors = []
        k_eff = min(self.k, len(distances))
        for i in range(k_eff):
            neighbors.append(distances[i][0])
        return neighbors

    def predict(self, test_set):
        predictions = []
        for test_sample in test_set:
            neighbors = self.nearest_neighbors(test_sample)
            labels = [sample for sample in neighbors]
            prediction = max(labels, key=labels.count)
            predictions.append(prediction)
        return np.array(predictions)

## Datasets disponíveis e seleção para teste empírico

In [ ]:
data_dir = Path("class_imbalance/class_imbalance")
all_csv = sorted(data_dir.glob("*.csv"))

print(f"Total de datasets encontrados: {len(all_csv)}")
for p in all_csv[:10]:
    print("-", p.name)

MAX_DATASETS = 12
selected_paths = all_csv[:MAX_DATASETS]

print("\nDatasets selecionados:")
for p in selected_paths:
    print("-", p.name, "| existe:", p.exists())

## Funções auxiliares para análise e avaliação

In [ ]:
def infer_target_column(df):
    return df.columns[-1]

def class_distribution_report(y, dataset_name, plot=False):
    s = pd.Series(y)
    counts = s.value_counts(dropna=False)
    ratios = (counts / len(s)).round(4)

    print(f"\n=== Distribuição de classes: {dataset_name} ===")
    dist_df = pd.DataFrame({"count": counts, "ratio": ratios})
    print(dist_df)

    if len(counts) > 1:
        imbalance_ratio = counts.max() / counts.min()
        print(f"Imbalance ratio (majoritária/minoritária): {imbalance_ratio:.2f}")
    else:
        print("Apenas uma classe presente.")

    if plot:
        ax = counts.sort_index().plot(kind="bar", figsize=(6, 3), title=f"Distribuição - {dataset_name}")
        ax.set_xlabel("Classe")
        ax.set_ylabel("Contagem")
        plt.tight_layout()
        plt.show()

def evaluate_predictions(y_true, y_pred, average_labels=("macro", "weighted")):
    out = {}
    out["accuracy"] = accuracy_score(y_true, y_pred)
    out["balanced_accuracy"] = balanced_accuracy_score(y_true, y_pred)

    for avg in average_labels:
        p, r, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average=avg, zero_division=0
        )
        out[f"precision_{avg}"] = p
        out[f"recall_{avg}"] = r
        out[f"f1_{avg}"] = f1

    return out

def is_dataset_valid_for_stratified_split(y, test_size=0.25):
    s = pd.Series(y)
    counts = s.value_counts(dropna=False)
    if len(counts) < 2:
        return False, "Apenas uma classe."
    if counts.min() < 2:
        return False, "Classe com menos de 2 amostras (split estratificado inviável)."
    n_test = int(np.ceil(len(s) * test_size))
    if n_test < len(counts):
        return False, "Conjunto de teste pequeno demais para conter todas as classes."
    return True, 

## Fase 2 — KNN com k Adaptativo (Variante 1: Estabilidade de Entropia)

### Motivação
O KNN standard usa um k fixo global escolhido por cross-validation. Com class imbalance,
este valor único é problemático:

- Com k grande, a classe minoritária é "engolida" pela maioritária em votação
- Num k fixo pequeno, zonas densas e homogéneas ficam sub-aproveitadas

### Proposta
Para cada ponto de teste, escolher dinamicamente o k ideal analisando a estrutura local
da sua vizinhança em múltiplas escalas (valores de k crescentes).

**Critério de paragem — Estabilidade de Entropia:**
Calculamos H(k) = entropia de Shannon das classes na k-vizinhança para vários valores de k.
O k* escolhido é aquele onde a variação de entropia entre escalas consecutivas é mínima —
ou seja, onde a estrutura local "estabiliza".

- Em zonas homogéneas (longe de fronteiras): H estabiliza rapidamente com k grande → k* grande
- Em zonas de fronteira (classes misturadas): H mantém-se alta mesmo com k pequeno → k* pequeno, protegendo a classe minoritária

**Limite superior de k:** k_max ≤ min(√n_treino, n_minoritária − 1), garantindo que a
classe minoritária pode sempre vencer uma votação.

In [ ]:
class AdaptiveKNN:
    """
    KNN com k adaptativo por ponto, baseado em estabilidade de entropia.

    Para cada ponto de teste:
      1. Ordena todos os pontos de treino por distância euclidiana.
      2. Calcula H(k) = entropia de Shannon das classes na k-vizinhança,
         para k em {k_min, ..., k_max} amostrado em n_scales pontos.
      3. Escolhe k* onde |H(k_i) - H(k_{i-1})| é mínimo (ponto de estabilização).
      4. Prediz por maioria simples com os k* vizinhos mais próximos.

    Vantagem sobre KNN standard em datasets desbalanceados:
      - k_max é limitado por n_minoritária - 1, impedindo que a classe rara
        seja sempre superada em número.
      - Zonas de fronteira (alta entropia) forçam k pequeno, dando voz
        à classe minoritária mesmo em regiões mistas.
    """

    def __init__(self, k_min: int = 3, k_max: int | None = None, n_scales: int = 8):
        self.k_min   = k_min
        self.k_max   = k_max
        self.n_scales = n_scales

    # ------------------------------------------------------------------
    def fit(self, X_train, y_train):
        self.x_train = np.array(X_train, dtype=float)
        self.y_train = np.array(y_train)

        classes, counts = np.unique(self.y_train, return_counts=True)
        n_minority = int(counts.min())
        n_train    = len(self.y_train)

        if self.k_max is None:
            # Upper bound: cannot exceed minority class size - 1
            # (so minority can always win a majority vote)
            self.k_max = max(self.k_min + 1,
                             min(int(np.sqrt(n_train)), n_minority - 1))

        self.k_max = min(self.k_max, n_train - 1)
        self.k_max = max(self.k_max, self.k_min + 1)

        self._classes = classes

    # ------------------------------------------------------------------
    @staticmethod
    def _entropy(labels) -> float:
        if len(labels) == 0:
            return 0.0
        _, counts = np.unique(labels, return_counts=True)
        p = counts / counts.sum()
        return float(-np.sum(p * np.log2(p + 1e-12)))

    # ------------------------------------------------------------------
    def _find_adaptive_k(self, sorted_labels) -> int:
        n_avail   = len(sorted_labels)
        k_max_eff = min(self.k_max, n_avail)
        k_min_eff = min(self.k_min, k_max_eff)

        k_values = np.unique(
            np.linspace(k_min_eff, k_max_eff, self.n_scales).astype(int)
        )
        k_values = k_values[k_values >= 1]

        if len(k_values) <= 1:
            return int(k_values[0]) if len(k_values) else k_min_eff

        entropies = np.array([
            self._entropy(sorted_labels[:k]) for k in k_values
        ])

        # Point of minimum entropy change between consecutive scales
        entropy_changes = np.abs(np.diff(entropies))
        stable_idx = int(np.argmin(entropy_changes))

        return int(k_values[stable_idx])

    # ------------------------------------------------------------------
    def _sorted_labels(self, test_sample):
        """Return training labels sorted by distance to test_sample (vectorised)."""
        diffs = self.x_train - test_sample          # (n_train, d)
        dists = np.sqrt((diffs ** 2).sum(axis=1))   # (n_train,)
        return self.y_train[np.argsort(dists)]

    # ------------------------------------------------------------------
    def predict(self, test_set, return_k_values: bool = False):
        predictions = []
        k_chosen    = []

        for sample in test_set:
            sl     = self._sorted_labels(sample)
            k_star = self._find_adaptive_k(sl)
            nbrs   = list(sl[:k_star])
            predictions.append(max(set(nbrs), key=nbrs.count))
            k_chosen.append(k_star)

        preds = np.array(predictions)
        return (preds, np.array(k_chosen)) if return_k_values else preds


## Avaliação empírica comparativa: KNN Standard vs. KNN Adaptativo

Corremos quatro modelos em cada dataset:
- **knn_from_scratch** — implementação base (k=5, voto por maioria)
- **sklearn_knn_uniform** — sklearn com pesos uniformes
- **sklearn_knn_distance** — sklearn com pesos por distância
- **knn_adaptive_v1** — *nossa proposta*: k adaptativo por estabilidade de entropia

In [ ]:
results      = []
skipped      = []
k_dist_store = {}   # k values chosen per dataset by AdaptiveKNN

for ds_path in selected_paths:
    if not ds_path.exists():
        skipped.append({"dataset": ds_path.name, "reason": "Ficheiro não encontrado"})
        continue

    print("\n" + "=" * 80)
    print(f"Dataset: {ds_path.name}")

    try:
        df = pd.read_csv(ds_path)
        if df.shape[1] < 2:
            skipped.append({"dataset": ds_path.name, "reason": "Colunas insuficientes"})
            continue

        target_col = infer_target_column(df)
        X = df.drop(columns=[target_col]).copy()
        y = df[target_col].copy()

        mask = ~pd.isna(y)
        X, y = X.loc[mask], y.loc[mask]

        X = pd.get_dummies(X, drop_first=True)
        if X.isnull().any().any():
            X = X.fillna(X.median(numeric_only=True))
        X = X.apply(pd.to_numeric, errors="coerce").fillna(0.0)

        if y.dtype == "O" or str(y.dtype).startswith("category"):
            y = y.astype("category").cat.codes

        valid, reason = is_dataset_valid_for_stratified_split(y, test_size=0.25)
        if not valid:
            skipped.append({"dataset": ds_path.name, "reason": reason})
            print(f"Ignorado: {reason}")
            continue

        class_distribution_report(y, ds_path.name, plot=False)

        X_arr = X.values
        y_arr = np.array(y)

        X_train, X_test, y_train, y_test = train_test_split(
            X_arr, y_arr, test_size=0.25, random_state=42, stratify=y_arr
        )

        sc = StandardScaler()
        X_train_sc = sc.fit_transform(X_train)
        X_test_sc  = sc.transform(X_test)

        k_eff = min(5, len(X_train))

        # ── 1. KNN from scratch (standard) ────────────────────────────
        knn_s = KNN(k=k_eff)
        knn_s.fit(X_train_sc, y_train)
        pred_s = knn_s.predict(X_test_sc)
        met = evaluate_predictions(y_test, pred_s)
        met.update({"dataset": ds_path.name, "model": "knn_from_scratch"})
        results.append(met)

        # ── 2. sklearn uniform ─────────────────────────────────────────
        knn_u = KNeighborsClassifier(n_neighbors=k_eff, weights="uniform")
        knn_u.fit(X_train_sc, y_train)
        met = evaluate_predictions(y_test, knn_u.predict(X_test_sc))
        met.update({"dataset": ds_path.name, "model": "sklearn_knn_uniform"})
        results.append(met)

        # ── 3. sklearn distance ────────────────────────────────────────
        knn_d = KNeighborsClassifier(n_neighbors=k_eff, weights="distance")
        knn_d.fit(X_train_sc, y_train)
        met = evaluate_predictions(y_test, knn_d.predict(X_test_sc))
        met.update({"dataset": ds_path.name, "model": "sklearn_knn_distance"})
        results.append(met)

        # ── 4. Adaptive KNN (Variante 1) ───────────────────────────────
        knn_a = AdaptiveKNN(k_min=3, n_scales=8)
        knn_a.fit(X_train_sc, y_train)
        pred_a, k_vals = knn_a.predict(X_test_sc, return_k_values=True)
        k_dist_store[ds_path.name] = k_vals
        print(f"  [AdaptiveKNN] k* — mean: {k_vals.mean():.1f}  "
              f"std: {k_vals.std():.1f}  "
              f"min: {k_vals.min()}  max: {k_vals.max()}")
        met = evaluate_predictions(y_test, pred_a)
        met.update({"dataset": ds_path.name, "model": "knn_adaptive_v1"})
        results.append(met)

    except Exception as e:
        skipped.append({"dataset": ds_path.name, "reason": f"Erro: {e}"})
        print(f"Ignorado por erro: {e}")


## Análise dos valores de k* escolhidos pelo AdaptiveKNN

Verificamos como o k adaptativo varia por dataset e por ponto — isto revela se o modelo
está realmente a adaptar-se à estrutura local ou a convergir sempre para o mesmo k.

In [ ]:
if k_dist_store:
    fig, axes = plt.subplots(
        nrows=max(1, len(k_dist_store) // 3 + (1 if len(k_dist_store) % 3 else 0)),
        ncols=min(3, len(k_dist_store)),
        figsize=(14, 3 * max(1, len(k_dist_store) // 3 + 1)),
        squeeze=False,
    )
    axes_flat = axes.flatten()

    for ax, (ds_name, k_vals) in zip(axes_flat, k_dist_store.items()):
        ax.hist(k_vals, bins=range(int(k_vals.min()), int(k_vals.max()) + 2),
                color="steelblue", edgecolor="white", alpha=0.85)
        ax.set_title(ds_name.replace("dataset_", "").replace(".csv", ""),
                     fontsize=8)
        ax.set_xlabel("k* escolhido")
        ax.set_ylabel("# pontos de teste")
        ax.axvline(k_vals.mean(), color="crimson", linestyle="--",
                   linewidth=1.2, label=f"média={k_vals.mean():.1f}")
        ax.legend(fontsize=7)

    for ax in axes_flat[len(k_dist_store):]:
        ax.set_visible(False)

    fig.suptitle("Distribuição dos k* adaptativos por dataset", fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum resultado do AdaptiveKNN disponível.")


## Tabela final de resultados e ranking global

In [ ]:
results_df = pd.DataFrame(results)

metric_cols = [
    "accuracy", "balanced_accuracy",
    "precision_macro", "recall_macro", "f1_macro",
    "precision_weighted", "recall_weighted", "f1_weighted",
]

if not results_df.empty:
    display_df = (
        results_df[["dataset", "model"] + metric_cols]
        .sort_values(by=["dataset", "balanced_accuracy"], ascending=[True, False])
    )
    print("\n=== Resultados por dataset/modelo ===")
    print(display_df.to_string(index=False))

    ranking = (
        results_df
        .groupby("model")[["balanced_accuracy", "f1_macro", "f1_weighted"]]
        .mean()
        .sort_values(by=["balanced_accuracy", "f1_macro"], ascending=False)
        .reset_index()
    )
    print("\n=== Ranking médio global por modelo ===")
    print(ranking.to_string(index=False))

    # Highlight: AdaptiveKNN vs KNN from scratch
    pivot = results_df.pivot_table(
        index="dataset", columns="model", values="balanced_accuracy"
    )
    if "knn_adaptive_v1" in pivot.columns and "knn_from_scratch" in pivot.columns:
        pivot["delta_adaptive_vs_scratch"] = (
            pivot["knn_adaptive_v1"] - pivot["knn_from_scratch"]
        )
        print("\n=== Delta balanced_accuracy: AdaptiveKNN − KNN Standard ===")
        print(pivot[["knn_from_scratch", "knn_adaptive_v1",
                      "delta_adaptive_vs_scratch"]].to_string())
else:
    print("Sem resultados. Verifica os datasets selecionados.")

if skipped:
    print("\n=== Datasets ignorados ===")
    print(pd.DataFrame(skipped).to_string(index=False))
